In [5]:
import requests
import API
import json
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, FloatType, IntegerType

In [2]:
def extract(cities):

        url = f"http://api.openweathermap.org/data/2.5/forecast?q={cities}&appid={API.weather_api_key}&units=metric"
        weather_data_response = requests.get(url)

        try:
            extracted_data = weather_data_response.json()
            
        except json.JSONDecodeError as e:
                print(f"error in data handling {e}")

        return extracted_data

In [3]:
def transform(data_to_transform):

    def get_solar_panel_status(temp):
        if temp >= 95:
            return "Exceptional"
        elif temp >= 80:
            return "High"
        elif temp >= 60:
            return "Average"
        elif temp >= 40:
            return "Budget"
        else:
            return "Sub-par"
        
    def get_wind_direction(deg):
        if deg is None:
            return "Unknown"
        
        deg = deg % 360 
        
        if 337.5 <= deg or deg < 22.5:
            return "North"
        elif 22.5 <= deg < 67.5:
            return "North-East"
        elif 67.5 <= deg < 112.5:
            return "East"
        elif 112.5 <= deg < 157.5:
            return "South-East"
        elif 157.5 <= deg < 202.5:
            return "South"
        elif 202.5 <= deg < 247.5:
            return "South-West"
        elif 247.5 <= deg < 292.5:
            return "West"
        elif 292.5 <= deg < 337.5:
            return "North-West"

    transformed_data = pd.json_normalize(
        data_to_transform,
        record_path=["list"],
        meta=[
            ["city", "id"],
            ["city", "name"],
            ["city", "country"],
            ["city", "population"],
        ],
        errors="ignore"
    )

    transformed_data['weather_main'] = transformed_data['weather'].apply(
        lambda x: x[0]['main'] if isinstance(x, list) else None
    )

    transformed_data['weather_desc'] = transformed_data['weather'].apply(
        lambda x: x[0]['description'] if isinstance(x, list) else None
    )

    transformed_data = transformed_data.rename(columns={
        "dt_txt": "datetime",
        "main.temp": "temperature",
        "main.temp_min": "temp_min",
        "main.temp_max": "temp_max",
        "main.pressure": "pressure",
        "main.sea_level": "sea_level",
        "main.humidity": "humidity",
        "city.name": "city",
        "city.id": "city_id",
        "city.country": "country",
        "city.population": "population",
        "clouds.all" : "cloud_percentage",
        "wind.speed" : "wind_speed",
        "wind.deg" : "wind_degrees",
        "wind.gust" : "gust"
    })
    

    transformed_data["temp_fahrenheit"] = (transformed_data["temperature"] * 1.8) + 32

    transformed_data["status"] = transformed_data["temperature"].apply(
        lambda x: "Very Hot" if x > 35 else ("warm" if x > 25 else "cool")
    )

    transformed_data["cloud_label"] = transformed_data["cloud_percentage"].apply(
        lambda x: "Sunny" if x <= 20 else ("Partly Cloudy" if x <= 60 else "Overcast")
    )

    transformed_data["Solar_Power_Level"] = transformed_data["temperature"].apply(get_solar_panel_status)

    transformed_data["wind_direction"] = transformed_data["wind_degrees"].apply(get_wind_direction) 

    cleaned_data = transformed_data[[
        "city", "country",
        "datetime", "weather_main",
        "temperature",
        "temp_fahrenheit", "status",
        "cloud_percentage", "cloud_label",
        "Solar_Power_Level", "wind_speed",
        "wind_degrees", "wind_direction",
        "weather_desc", "city_id",
        "humidity", "temp_min",
        "temp_max", "pressure"
    ]]
    
    return cleaned_data

In [ ]:
def load(loaded_sparkdata):
    
    spark = (
        SparkSession.builder
            .master("local[*]")
            .appName("ELT-Batch-Processing")
            .config("spark.jars", "./postgresql-42.7.9.jar")
            .getOrCreate()
    )

    schema = StructType([
        StructField("city", StringType(), False),
        StructField("country", StringType(), False),
        StructField("datetime", TimestampType(), False),
        StructField("weather_main", StringType(), False),
        StructField("temperature", FloatType(), False),
        StructField("temp_fahrenheit", FloatType(), False),
        StructField("status", StringType(), True),
        StructField("cloud_percentage", IntegerType(), True), 
        StructField("cloud_label", StringType(), False),
        StructField("Solar_Power_level", StringType(), True),
        StructField("wind_speed", FloatType(), False),
        StructField("wind_degrees", IntegerType(), False),
        StructField("wind_direction", StringType(), False),
        StructField("weather_desc", StringType(), False),
        StructField("city_id", IntegerType(), False),
        StructField("humidity", IntegerType(), False),
        StructField("temp_min", FloatType(), False),
        StructField("temp_max", FloatType(), False),
        StructField("pressure", IntegerType(), False)
    ])

    transformed_to_spark = spark.createDataFrame(loaded_sparkdata, schema=schema)

    db_url = "jdbc:postgresql://localhost:5432/WeatherDB"
    db_properties = {
        "user" : "postgrest",
        "password" : API.postgresql_password,
        "driver" : "org.postgresql.Driver"
    }

    transformed_to_spark.write.jdbc(url=db_url, table="weather_report", mode="append", properties=db_properties)

    print("succesfully loaded to postgresql")

    spark.stop()
    

In [11]:
city = "Manila"
data_extracted_form = extract(city)
data_transformed = transform(data_extracted_form)

In [12]:
data_transformed

,city,country,datetime,weather_main,temperature,temp_fahrenheit,status,cloud_percentage,cloud_label,Solar_Power_Level,wind_speed,wind_degrees,wind_direction,weather_desc,city_id,humidity,temp_min,temp_max,pressure
0,Manila,PH,2026-04-08 18:00:00,Clouds,27.39,81.302,warm,13,Sunny,Sub-par,2.73,97,East,few clouds,1701668,63,26.08,27.39,1013
1,Manila,PH,2026-04-08 21:00:00,Clouds,25.97,78.746,warm,11,Sunny,Sub-par,2.11,63,North-East,few clouds,1701668,62,24.93,25.97,1012
2,Manila,PH,2026-04-09 00:00:00,Clear,28.61,83.498,warm,5,Sunny,Sub-par,1.49,12,North,clear sky,1701668,55,28.61,28.61,1014
3,Manila,PH,2026-04-09 03:00:00,Clear,31.64,88.952,warm,2,Sunny,Sub-par,3.11,242,South-West,clear sky,1701668,50,31.64,31.64,1013
4,Manila,PH,2026-04-09 06:00:00,Clouds,35.02,95.036,Very Hot,12,Sunny,Sub-par,3.20,165,South,few clouds,1701668,50,35.02,35.02,1010
5,Manila,PH,2026-04-09 09:00:00,Clouds,32.30,90.140,warm,32,Partly Cloudy,Sub-par,6.04,132,South-East,scattered clouds,1701668,55,32.30,32.30,1009
6,Manila,PH,2026-04-09 12:00:00,Clouds,29.49,85.082,warm,26,Partly Cloudy,Sub-par,6.37,142,South-East,scattered clouds,1701668,66,29.49,29.49,1011
7,Manila,PH,2026-04-09 15:00:00,Rain,28.16,82.688,warm,24,Partly Cloudy,Sub-par,2.83,88,East,light rain,1701668,67,28.16,28.16,1013
8,Manila,PH,2026-04-09 18:00:00,Clouds,27.30,81.140,warm,34,Partly Cloudy,Sub-par,2.68,109,East,scattered clouds,1701668,71,27.30,27.30,1011
9,Manila,PH,2026-04-09 21:00:00,Clear,26.78,80.204,warm,4,Sunny,Sub-par,1.63,74,East,clear sky,1701668,72,26.78,26.78,1011
